# Prever o retorno de investimentos com Temporal Fusion Transformers

In [2]:
# Instala o pacote watermark. 
!pip install -q -U watermark

In [3]:
%env TF_CPP_MIN_LOG_LEVEL=3

env: TF_CPP_MIN_LOG_LEVEL=3


In [4]:
# https://www.tensorflow.org/
!pip install -q tensorflow

In [5]:
# https://pypi.org/project/ta/
!pip install -q ta

In [6]:
# https://pypi.org/project/yfinance/
!pip install -q yfinance

In [7]:
# Imports
import ta
import sklearn
import pandas as pd
import numpy as np
import tensorflow
import yfinance as yf
import matplotlib.pyplot as plt
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import warnings
warnings.filterwarnings("ignore")

## Extração dos dados

In [14]:
def download_data(ticker, start = "2000-01-01", end = "2024-12-31"):
    
    # Força que venha 'Adj Close' definindo auto_adjust=False
    dados = yf.download(ticker, start=start, end=end, auto_adjust=False)

    # Mapeamento dos nomes originais para minúsculas e underscore
    mapeamento = {
        'Open':       'open',
        'High':       'high',
        'Low':        'low',
        'Close':      'close',
        'Adj Close':  'adj_close',
        'Volume':     'volume'
    }

    presentes = {col: novo for col, novo in mapeamento.items() if col in dados.columns}
    dados.rename(columns = presentes, inplace = True)

    dados.index.name = "date"
    
    return dados

In [28]:
# Extração dos dados
df = download_data("MSFT")

[*********************100%***********************]  1 of 1 completed


In [20]:
df.head()

Price,adj_close,close,high,low,open,volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,MSFT
date,,,,,,
2000-01-03,35.601452,58.28125,59.3125,56.00000,58.68750,53228400
2000-01-04,34.398811,56.31250,58.5625,56.12500,56.78125,54119000
2000-01-05,34.761528,56.90625,58.1875,54.68750,55.56250,64059600
2000-01-06,33.597073,55.00000,56.9375,54.18750,56.09375,54976600
2000-01-07,34.036140,55.71875,56.1250,53.65625,54.31250,62013600


In [21]:
df.tail()

Price,adj_close,close,high,low,open,volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,MSFT
date,,,,,,
2023-12-22,369.077087,374.579987,375.179993,372.709991,373.679993,17107500
2023-12-26,369.155914,374.660004,376.940002,373.500000,375.000000,12673100
2023-12-27,368.574615,374.070007,375.059998,372.809998,373.690002,14905400
2023-12-28,369.766846,375.279999,376.459991,374.160004,375.369995,14327000
2023-12-29,370.515686,376.040009,377.160004,373.480011,376.000000,18730800


## Engenharia de Atributos

Pacote `ta`: Technical Analysis Library in Python

https://github.com/bukosabino/ta

Criaremos uma função de engenharia de atributos para um dataframe que representa dados de um ativo financeiro (ações) com colunas para 'open' (preço de abertura), 'high' (máximo do dia), 'low' (mínimo do dia), 'close' (preço de fechamento) e 'volume'. 

A engenharia de atributos é uma técnica usada para criar novas variáveis com base em variáveis existentes, a fim de melhorar o desempenho dos modelos de Machine Learning. 

In [36]:
# Função para engenharia de atributos com dados coletados na versão mais recente do yfinance
def func_engenharia_atributos(df):

    df_copy = df.copy()
    
    # Se houver MultiIndex nas colunas, mantém só o primeiro nível
    if isinstance(df_copy.columns, pd.MultiIndex):
        df_copy.columns = df_copy.columns.get_level_values(0)
    
    # Cria a variável com o retorno (mudança percentual do fechamento - close)
    # Essa será nossa variável alvo
    df_copy["retorno"] = df_copy["close"].pct_change(1)
    
    # Shift das colunas de preço do ativo financeiro
    df_copy["op"]  = df_copy["open"].shift(1)
    df_copy["hi"]  = df_copy["high"].shift(1)
    df_copy["lo"]  = df_copy["low"].shift(1)
    df_copy["clo"] = df_copy["close"].shift(1)

    # Shift da coluna Volume
    df_copy["vol"] = df_copy["volume"].shift(1)
    
    # Simple Moving Average (SMA)
    df_copy["SMA_15"] = df_copy["close"].rolling(15).mean().shift(1)
    df_copy["SMA_60"] = df_copy["close"].rolling(60).mean().shift(1)

    # Moving Standard Deviation (MSD) - Volatilidade
    df_copy["MSD_15"] = df_copy["retorno"].rolling(15).std().shift(1)
    df_copy["MSD_60"] = df_copy["retorno"].rolling(60).std().shift(1)
    
    # Volume Weighted Average Price (VWAP) -  Média ponderada de preço com base no volume
    vwap = ta.volume.VolumeWeightedAveragePrice(high   = df_copy["high"],
                                                low    = df_copy["low"],
                                                close  = df_copy["close"],
                                                volume = df_copy["volume"],
                                                window = 5)
    
    df_copy["VWAP"] = vwap.vwap.shift(1)
    
    # RSI - Índice de Força Relativa 
    # (agora df_copy["close"] é Series 1D)
    rsi = ta.momentum.RSIIndicator(df_copy["close"], window = 5, fillna = False)
    df_copy["RSI"] = rsi.rsi().shift(1)
    
    return df_copy.dropna()

`func_engenharia_atributos`

- Cria uma cópia do dataframe para não modificar o dataframe original.

- Cria uma nova coluna 'retorno', que representa a mudança percentual do preço de fechamento em relação ao dia anterior.

- As colunas 'open', 'high', 'low', 'close' e 'volume' são então deslocadas para baixo (shifted) em uma unidade, criando as colunas 'op', 'hi', 'lo', 'clo' e 'vol'. Isso significa que a linha 'i' dessas novas colunas contém os valores da linha 'i-1' das colunas originais. Isso é chamado de defasagem e amplamente usado na modelagem de séries temporais.

- Cria **médias móveis simples** (Simple Moving Average - **SMA**) para o preço de fechamento com janelas de 15 e 60 dias, deslocadas por uma unidade. 

- A **volatilidade**, representada como o desvio padrão das mudanças percentuais do preço de fechamento, é calculada para as janelas de 15 e 60 dias, também deslocadas por uma unidade. Esse índice é conhecido como Moving Standard Deviation (**MSD**).

- Cria o Volume Weighted Average Price (**VWAP**) com uma janela de 5 dias. O VWAP é uma **medida de preço médio ponderado pelo volume**.

- Adiciona o indicador **RSI** (Relative Strength Index) com uma janela de 5 dias. O RSI é um **indicador de momento que mede a velocidade e a mudança de movimentos de preço**.

- Remove todas as linhas que contêm valores NA (que foram criados ao deslocar as colunas e calcular médias móveis e RSI com janelas).

Em resumo, essa função cria várias novas características técnicas comumente usadas na análise de ativos financeiros, todas deslocadas por uma unidade, para evitar o uso de informações futuras (ou seja, vazamento de dados) no modelo de Machine Learning.

In [37]:
# Engenharia de atributos
df2 = func_engenharia_atributos(df)

In [38]:
df2.columns

Index(['adj_close', 'close', 'high', 'low', 'open', 'volume', 'retorno', 'op',
       'hi', 'lo', 'clo', 'vol', 'SMA_15', 'SMA_60', 'MSD_15', 'MSD_60',
       'VWAP', 'RSI'],
      dtype='object', name='Price')

## Pré-processamento de Dados

In [40]:
split = int(len(df2) * 0.8)
split_val = int(len(df2) * 0.95)

In [46]:
# Dataset de treino
x_treino = df2[['VWAP','RSI','SMA_15','SMA_60','MSD_15','MSD_60','op','hi','lo','clo','vol']].iloc[:split,:]
y_treino = df2[['retorno']].iloc[:split,:]

# Dataset de validação
x_valid = df2[['VWAP','RSI','SMA_15','SMA_60','MSD_15','MSD_60','op','hi','lo','clo','vol']].iloc[split:split_val,:]
y_valid = df2[['retorno']].iloc[split:split_val,:]

# Dataset de teste
x_teste = df2[['VWAP','RSI','SMA_15','SMA_60','MSD_15','MSD_60','op','hi','lo','clo','vol']].iloc[split_val:,:]
y_teste = df2[['retorno']].iloc[split_val:,:]

### Padronização

In [47]:
# Cria o padronizador
sc = StandardScaler()

In [50]:
# Fit e transform nos dados de treino
x_treino_sc = sc.fit_transform(x_treino)

# Transform nos dados de validação
x_valid_sc = sc.transform(x_valid)

# Transform nos dados de teste
x_teste_sc = sc.transform(x_teste)

### Ajuste no formato dos dados

O código abaixo realiza as seguintes operações:

Primeiro, verifica se as entradas X_s e y_s possuem a mesma extensão. Se não, imprime um aviso.

Em seguida, cria a variável X_train. Essa variável é criada iterando sobre todas as variáveis em X_s e criando 'janelas' de tamanho lag para cada uma delas. Ou seja, para cada ponto no tempo i, ele pega os lag pontos anteriores para cada variável e armazena isso em uma lista. Ao final do processo, temos uma lista de listas para cada variável.

X_train é então transformado em um array NumPy e as dimensões são reordenadas. A chamada ao método swapaxes é feita duas vezes para garantir que as dimensões estejam na ordem correta para a entrada do modelo. A ordem final é (amostras, passos de tempo, características).

Depois, a função cria y_train que contém a variável alvo correspondente para cada janela em X_train. Ele faz isso iterando sobre y_s de lag até o final e armazenando cada valor em uma lista.

Por fim, y_train é convertido em um array NumPy usando o método concatenate, que combina uma lista de arrays NumPy em um único array.

A função retorna X_train e y_train, que estão agora em um formato adequado para serem usados para treinar uma RNN. A ideia geral aqui é que, para prever o valor de y em um certo ponto no tempo, você usará os lag pontos anteriores de X.

In [51]:
x_treino.head(1)

Price,VWAP,RSI,SMA_15,SMA_60,MSD_15,MSD_60,op,hi,lo,clo,vol
date,,,,,,,,,,,
2000-03-30,54.201657,60.893094,50.891667,50.815625,0.038358,0.031934,52.59375,54.46875,52.5625,53.59375,64363800.0


In [55]:
x_treino_sc

array([[ 0.4158323 ,  0.42733862,  0.30561856, ...,  0.37239467,
         0.39065614,  0.29468238],
       [ 0.3942353 , -0.30606939,  0.30959594, ...,  0.3262237 ,
         0.32412396,  0.28882114],
       [ 0.36634582,  0.19055006,  0.31578296, ...,  0.35480573,
         0.37429577,  0.29207739],
       ...,
       [ 4.15193423,  1.11090904,  4.13425691, ...,  4.1961427 ,
         4.21893947, -1.05641731],
       [ 4.18124232,  0.50412515,  4.14538177, ...,  4.21232477,
         4.17880224, -0.99790284],
       [ 4.2102684 ,  0.80060264,  4.15898142, ...,  4.24222618,
         4.21544905, -1.06301201]])

In [58]:
# Função para ajustar o formato dos dados
def ajusta_formato_dados(X_s, y_s, lag):

    # Verifica se o comprimento de X_s é igual ao comprimento de y_s
    if len(X_s) != len(y_s):
        print("Warnings")

    # Inicializa a lista X_train
    X_train = []
    
    # Itera sobre cada variável (coluna) em X_s
    for variable in range(0, X_s.shape[1]):
        
        # Inicializa a lista X para a variável atual
        X = []
        
        # Cria sequências de tamanho 'lag' para a variável atual
        for i in range(lag, X_s.shape[0]):
            X.append(X_s[i-lag:i, variable])
        
        # Adiciona a lista de sequências para a variável atual em X_train
        X_train.append(X)
    
    # Converte X_train para um array numpy e ajusta os eixos
    X_train, np.array(X_train)
    X_train = np.swapaxes(np.swapaxes(X_train, 0, 1), 1, 2)

    # Inicializa a lista y_train
    y_train = []
    
    # Cria a lista de rótulos ajustada de acordo com 'lag'
    for i in range(lag, y_s.shape[0]):
        y_train.append(y_s[i, :].reshape(-1,1).transpose())
    
    # Concatena a lista de rótulos em um array numpy
    y_train = np.concatenate(y_train, axis = 0)
    
    # Retorna X_train e y_train ajustados
    return X_train, y_train

In [59]:
# Valor do Lag
lag = 15

In [72]:
# Aplica a função nos dados de treino
x_treino_final, y_treino_final = ajusta_formato_dados(x_treino_sc, y_treino.values, lag)

# Aplica a função nos dados de validação
x_valid_final, y_valid_final = ajusta_formato_dados(x_valid_sc, y_valid.values, lag)

# Aplica a função nos dados de teste
x_teste_final, y_teste_final = ajusta_formato_dados(x_teste_sc, y_teste.values, lag)

In [63]:
y_treino_final.shape

(4966, 1)

## Construção do Modelo Temporal Fusion Transformer

https://arxiv.org/abs/1912.09363

In [64]:
# Função do transformer encoder
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout = 0):
    
    # Normaliza as entradas com camadas de normalização
    x = layers.LayerNormalization(epsilon = 1e-6)(inputs)
    
    # Aplica a atenção multi-cabeça nas entradas
    x = layers.MultiHeadAttention(key_dim = head_size, num_heads = num_heads, dropout = dropout)(x, x, x)
    
    # Aplica Dropout na saída da camada de atenção
    x = layers.Dropout(dropout)(x)
    
    # Adiciona as entradas iniciais como conexão residual
    res = x + inputs

    # Normaliza a soma da conexão residual
    x = layers.LayerNormalization(epsilon = 1e-6)(res)
    
    # Aplica uma camada convolucional com ativação ReLU
    x = layers.Conv1D(filters = ff_dim, kernel_size = 1, activation = "relu")(x)
    
    # Aplica Dropout após a camada convolucional
    x = layers.Dropout(dropout)(x)
    
    # Aplica uma segunda camada convolucional
    x = layers.Conv1D(filters = inputs.shape[-1], kernel_size = 1)(x)
    
    # Retorna a soma da segunda camada convolucional com a conexão residual
    return x + res

A função `transformer_encoder` define um bloco de codificador de transformer, um componente-chave na arquitetura do Transformer, adaptado para utilizar camadas convolucionais em vez de camadas totalmente conectadas. Aqui está uma breve explicação de cada passo da função:

**Normalização de Camada**: A função começa normalizando as entradas usando layers.LayerNormalization. A normalização é feita para estabilizar a aprendizagem, acelerando a convergência ao normalizar as saídas de cada camada para ter uma média zero e variância unitária.

**Atenção Multi-Cabeça**: Em seguida, a função aplica atenção multi-cabeça (layers.MultiHeadAttention) nas entradas normalizadas. A atenção multi-cabeça permite que o modelo entenda diferentes partes da entrada simultaneamente, melhorando sua capacidade de capturar relações complexas dentro dos dados.

**Dropout**: Após a atenção, um dropout é aplicado para regularizar o modelo, ajudando a prevenir o overfitting. O dropout aleatoriamente desativa uma porcentagem dos neurônios durante o treinamento, o que ajuda a tornar o modelo mais robusto.

**Conexão Residual**: A saída da camada de dropout é somada com as entradas originais, formando uma conexão residual. Essa conexão ajuda a evitar o problema do desvanecimento do gradiente em redes profundas, permitindo que o gradiente flua diretamente através das camadas sem ser atenuado.

**Segunda Normalização de Camada**: A soma resultante é novamente normalizada para manter as propriedades de estabilização do treinamento.

**Camadas Convolucionais**: Duas camadas convolucionais são aplicadas sequencialmente. A primeira camada convolucional (Conv1D) utiliza uma função de ativação ReLU e é destinada a introduzir não-linearidades, permitindo ao modelo aprender representações mais complexas. A segunda camada convolucional transforma a saída para ter a mesma dimensão que as entradas originais.

**Dropout após Convolucional**: Um segundo dropout é aplicado após a primeira camada convolucional para continuar a regularização do modelo.

**Saída Final**: Finalmente, a saída da última camada convolucional é somada à conexão residual original e esse resultado é retornado como a saída do bloco do codificador. Isso completa outro passo na construção de representações profundas e ricas dos dados de entrada.

https://www.researchgate.net/publication/352448180_Temporal_Fusion_Transformers_for_interpretable_multi-horizon_time_series_forecasting

In [65]:
# Função de criação do modelo
def cria_modelo(input_shape, 
                head_size, 
                num_heads, 
                ff_dim, 
                num_transformer_blocks, 
                mlp_units, 
                dropout = 0, 
                mlp_dropout = 0):
    
    # Define a entrada do modelo com a forma especificada
    inputs = keras.Input(shape = input_shape)
    
    # Inicializa a entrada do modelo na variável x
    x = inputs
    
    # Adiciona uma camada LSTM com 10 unidades e retorna sequências
    x = layers.LSTM(10, return_sequences = True)(x)
    
    # Adiciona blocos de transformer ao modelo
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    # Adiciona uma camada GRU com 100 unidades e não retorna sequências
    x = layers.GRU(100, return_sequences = False)(x)
    
    # Adiciona uma camada Dropout com a taxa especificada
    x = layers.Dropout(mlp_dropout)(x)
    
    # Adiciona uma camada densa com unidades especificadas e ativação ReLU
    x = layers.Dense(mlp_units, activation = "relu")(x)
    
    # Define a camada de saída com 1 unidade (saída do modelo)
    outputs = layers.Dense(1)(x)
    
    # Retorna o modelo criado com as entradas e saídas definidas
    return keras.Model(inputs, outputs)

A função `cria_modelo` é responsável por construir um modelo de Deep Learning que combina diferentes tipos de camadas, incluindo LSTM, GRU e blocos do Transformer, para processar sequências temporais. Aqui está uma explicação detalhada de cada parte da função:

**Entrada do Modelo**: A função inicia definindo a entrada do modelo utilizando keras.Input, que especifica a forma (shape) dos dados de entrada que o modelo espera.

**Camada LSTM**: Após definir a entrada, a função adiciona uma camada LSTM (layers.LSTM) com 10 unidades. A opção return_sequences=True indica que a camada deve retornar a sequência completa de saídas para todas as unidades de tempo, o que é necessário para conectar a saída desta camada a outras que também operam em sequências.

**Blocos de Transformer**: A função então entra em um loop para adicionar um número especificado de blocos de Transformer (num_transformer_blocks) ao modelo. Cada bloco é adicionado chamando a função dsa_transformer_encoder, que aplica um conjunto de operações, incluindo atenção multi-cabeça e convoluções, para processar e transformar a sequência de entrada.

**Camada GRU**: Após os blocos de Transformer, a função adiciona uma camada GRU (layers.GRU) com 100 unidades. Diferente da LSTM, return_sequences=False é usado aqui, o que significa que apenas a saída do último passo temporal é retornada. Isso é útil para tarefas onde o resultado final depende de uma compreensão global da sequência inteira.

**Dropout**: Um dropout (layers.Dropout) é então aplicado para ajudar a prevenir o overfitting, usando a taxa de dropout especificada em mlp_dropout. O dropout desativa aleatoriamente uma porção dos neurônios durante o treinamento, o que ajuda a fazer o modelo ser mais robusto e menos propenso a memorizar os dados de treinamento.

**Camada Densa com Ativação ReLU**: Uma camada densa (layers.Dense) com um número de unidades especificado em mlp_units e ativação ReLU é adicionada. Esta camada densa serve para consolidar as informações aprendidas pelas camadas anteriores em um formato mais compacto e é comumente usada para introduzir não-linearidades adicionais ao modelo.

**Camada de Saída**: Finalmente, uma camada densa com uma única unidade é adicionada como a camada de saída do modelo. Esta camada não especifica uma função de ativação, indicando que ela é destinada a uma tarefa de regressão.

**Construção do Modelo**: A função conclui retornando o modelo composto pelas entradas e saídas definidas, usando keras.Model. Isso encapsula toda a arquitetura em um objeto que pode ser treinado e utilizado para fazer previsões.

In [67]:
# Shape de entrada
input_shape = x_treino_final.shape[1:]

In [96]:
# Cria o modelo
modelo = cria_modelo(input_shape,
                     head_size = 16,
                     num_heads = 1,
                     ff_dim = 4,
                     num_transformer_blocks = 1,
                     mlp_units = 125,
                     dropout = 0.1,
                     mlp_dropout = 0.25)

In [97]:
# Compila o modelo
modelo.compile(loss = "mean_squared_error", optimizer = keras.optimizers.Adam())

In [ ]:
modelo.summary()

In [98]:
# Callbacks
callbacks = [keras.callbacks.EarlyStopping(patience = 5, restore_best_weights = True)]

### Treinamento e Avaliação

In [99]:
%%time

modelo.fit(x_treino_final,
           y_treino_final,
           validation_data = (x_valid_final, y_valid_final),
           epochs = 20,
           batch_size = 64,
           callbacks = callbacks)

Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - loss: 0.0219 - val_loss: 5.3526e-04
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 0.0014 - val_loss: 5.3072e-04
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 9.0758e-04 - val_loss: 5.4206e-04
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 7.2937e-04 - val_loss: 4.6582e-04
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 6.4155e-04 - val_loss: 5.5377e-04
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 5.6714e-04 - val_loss: 4.5855e-04
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 5.5137e-04 - val_loss: 5.4050e-04
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 5.3087e-04 - val_loss: 4.5089e-04
Epoch 9/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 5.3476e-04 - val_loss: 4.5384e-04
Epoch 10/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 4.7967e-04 - val_loss: 5.2593e-04
Epoch 11/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 4.5479e-04 - val_

In [ ]:
pd.DataFrame(modelo.history.history).plot()

In [101]:
# Previsões
pred = modelo.predict(x_teste_final)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step


In [102]:
# Calcula o score
score = np.sqrt(metrics.mean_squared_error(pred, y_teste_final))

In [103]:
print("Score (RMSE): {}".format(score))

Score (RMSE): 0.013076623745314582


## Previsões e Cálculo do Retorno

A linha de código abaixo está realizando a seguinte operação:

- Está prevendo valores para a entrada x_treino_final. Essa previsão retorna um array NumPy.

- Está criando um array NumPy de zeros com dimensões [lag,1]. A variável lag é geralmente um número inteiro que se refere a um período de atraso ou defasagem (por exemplo, se você está trabalhando com séries temporais e deseja considerar os 5 valores anteriores, o lag seria 5).

- Depois disso, está concatenando o array de zeros com as previsões do modelo ao longo do eixo 0 (ou seja, verticalmente). A razão para fazer isso é para ajustar o formato dos dados, porque você teve um deslocamento de lag ao criar os dados de entrada para o modelo, então agora precisa preencher esses primeiros pontos nos dados de saída.

In [ ]:
y_pred_treino = np.concatenate((np.zeros([lag,1]), modelo.predict(x_treino_final)), axis = 0)

In [ ]:
y_pred_valid = np.concatenate((np.zeros([lag,1]), modelo.predict(x_valid_final)), axis = 0)

In [ ]:
y_pred_teste = np.concatenate((np.zeros([lag,1]), modelo.predict(x_teste_final)), axis = 0)

In [90]:
# Concatena as previsões em treino, valid e teste como previsão final
df2["prediction"] = np.concatenate((y_pred_treino, y_pred_valid, y_pred_teste), axis = 0)

In [105]:
df2[["retorno", "prediction"]].tail(3)

Price,retorno,prediction
date,,
2024-12-26,-0.002777,0.002090
2024-12-27,-0.017302,0.001616
2024-12-30,-0.013240,0.001996


In [110]:
# Calcula a estratégia
df2["estrategia"] = df2["retorno"] * np.sign(df2["prediction"].shift(1))

A função np.sign retorna -1 se o número for negativo, 0 se o número for zero e 1 se o número for positivo.

Ao usar shift(1), você está pegando os valores de "prediction" do dia anterior, porque shift(1) desloca os valores para baixo em uma linha.

Em outras palavras, a estratégia aqui é que se a previsão do dia anterior era positiva, então você assume uma posição longa (compra) e seu retorno é simplesmente o retorno do ativo. Se a previeu retorno é o negativo do retorno do ativo (porque quando você vende um ativo a descosão do dia anterior era negativa, então você assume uma posição curta (vende), e o sberto, você ganha dinheiro quando o preço cai).